## 1. Setup and Imports

In [1]:
import os
import sys
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import json

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForTokenClassification
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import confusion_matrix, classification_report

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
	print(f"CUDA device: {torch.cuda.get_device_name(1)}")

# Set device
device = "cuda:1" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

PyTorch version: 2.5.1
CUDA available: True
CUDA device: Tesla P40

Using device: cuda:1


In [3]:
cd /home/dmoi/projects/ESM3di/

/home/dmoi/projects/ESM3di


## 2. Utility Functions

In [4]:
def read_fasta(path: str) -> List[Tuple[str, str]]:
	"""
	Simple FASTA parser.
	Returns list of (header_without_>, sequence_string).
	"""
	records = []
	header = None
	seq_chunks = []
	with open(path) as f:
		for line in f:
			line = line.strip()
			if not line:
				continue
			if line.startswith(">"):
				if header is not None:
					records.append((header, "".join(seq_chunks)))
				header = line[1:].strip()
				seq_chunks = []
			else:
				seq_chunks.append(line.strip().upper())
		if header is not None:
			records.append((header, "".join(seq_chunks)))
	return records

print("✓ FASTA reader function defined")

✓ FASTA reader function defined


In [55]:

def load_checkpoint(checkpoint_path: str, device: str = 'cpu'):
	"""
	Load a model checkpoint and return model, tokenizer, and metadata.
	
	Returns:
		model, tokenizer, label_vocab, mask_chars, config
	"""
	print(f"Loading checkpoint: {checkpoint_path}")
	
	ckpt = torch.load(checkpoint_path, map_location=device)
	# Extract metadata
	label_vocab = ckpt['label_vocab']
	mask_chars = ckpt.get('mask_label_chars', '')
	config = ckpt['args']
	
	epoch = ckpt.get('epoch', 0)
	loss = ckpt.get('loss', 0.0)

	print(f"  Checkpoint: epoch {epoch}, loss: {loss:.4f}")
	print(f"  Label vocab size: {len(label_vocab)}")
	print(f"  Mask characters: {mask_chars}")
	print(f"  Base model: {config['hf_model']}")

	# Get HF model name from config
	if isinstance(config, dict):
		hf_model = config.get('hf_model', 'facebook/esm2_t12_35M_UR50D')
		lora_r = config.get('lora_r', 8)
		lora_alpha = config.get('lora_alpha', 16.0)
		lora_dropout = config.get('lora_dropout', 0.05)
		lora_target_modules = config.get('lora_target_modules', None)
	else:
		hf_model = getattr(config, 'hf_model', 'facebook/esm2_t12_35M_UR50D')
		lora_r = getattr(config, 'lora_r', 8)
		lora_alpha = getattr(config, 'lora_alpha', 16.0)
		lora_dropout = getattr(config, 'lora_dropout', 0.05)
		lora_target_modules = getattr(config, 'lora_target_modules', None)
	print( 'loading fine tuned ', hf_model )
	# Load tokenizer
	tokenizer = AutoTokenizer.from_pretrained(hf_model)
	
	model = ESM3DiModel(
        hf_model_name=hf_model,
        num_labels=len(label_vocab),
        lora_r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=lora_target_modules
    )

	model = model.get_model()
	model.load_state_dict(ckpt['model_state_dict'])
	model.to(device)
	model.eval()
	
	print("✓ Checkpoint loaded successfully\n")
	
	return model, tokenizer, label_vocab, mask_chars, config

print("✓ Checkpoint loader function defined")

✓ Checkpoint loader function defined


In [56]:
@torch.no_grad()
def predict_3di_batch(model, tokenizer, label_vocab, aa_sequences: List[str], 
					 device: str = 'cpu', batch_size: int = 8) -> List[str]:
	"""
	Predict 3Di sequences for a batch of amino acid sequences.
	
	Args:
		model: Trained model
		tokenizer: ESM tokenizer
		label_vocab: List of 3Di characters
		aa_sequences: List of amino acid sequence strings
		device: Device to run on
		batch_size: Number of sequences to process at once
		
	Returns:
		List of predicted 3Di sequence strings
	"""
	model.eval()
	idx2char = {i: c for i, c in enumerate(label_vocab)}
	
	all_predictions = []
	
	# Process in batches
	for i in range(0, len(aa_sequences), batch_size):
		batch_seqs = aa_sequences[i:i+batch_size]
		
		# Tokenize batch
		enc = tokenizer(
			batch_seqs,
			return_tensors="pt",
			padding=True,
			truncation=True,
			add_special_tokens=True,
			return_special_tokens_mask=True,
		)
		enc = {k: v.to(device) for k, v in enc.items()}
		
		# Forward pass
		outputs = model(**enc)
		logits = outputs.logits  # [B, T, num_labels]
		special_mask = enc["special_tokens_mask"]  # [B, T]
		
		# Get predictions for each sequence in batch
		pred_indices = logits.argmax(dim=-1)  # [B, T]
		
		for b in range(len(batch_seqs)):
			pred_chars = []
			for j in range(pred_indices.shape[1]):
				if special_mask[b, j] == 1:
					# Skip special tokens
					continue
				pred_chars.append(idx2char[int(pred_indices[b, j])])
			
			all_predictions.append("".join(pred_chars))
	
	return all_predictions

print("✓ Batch prediction function defined")

✓ Batch prediction function defined


## 3. Configuration

Configure paths to your checkpoints and test data files.

In [57]:
# Evaluation configuration
EVAL_CONFIG = {
	# Checkpoint directory
	'checkpoint_dir': 'checkpoints',  # or 'checkpoints_notebook'
	
	# Which epochs to evaluate (None = all available)
	'epochs_to_eval': [1, 2, 3],  # or None for all
	
	# Test data paths
	'test_aa_fasta': '/mnt/data1/bfvd/training_set/bfvd_data_aa.fasta',
	'test_3di_fasta': '/mnt/data1/bfvd/training_set/bfvd_data_3di.fasta',  # Non-masked ground truth
	'test_3di_masked_fasta': '/mnt/data1/bfvd/training_set/bfvd_data_3di_masked.fasta',  # With masked positions
	
	# Evaluation settings
	'batch_size': 8,
	'mask_char': 'X',  # Character used for masking
	
	# Output
	'output_dir': 'evaluation_results',
}

# Create output directory
os.makedirs(EVAL_CONFIG['output_dir'], exist_ok=True)

print("Evaluation Configuration:")
for k, v in EVAL_CONFIG.items():
	print(f"  {k}: {v}")
print(f"\n✓ Output directory: {EVAL_CONFIG['output_dir']}")

Evaluation Configuration:
  checkpoint_dir: checkpoints
  epochs_to_eval: [1, 2, 3]
  test_aa_fasta: /mnt/data1/bfvd/training_set/bfvd_data_aa.fasta
  test_3di_fasta: /mnt/data1/bfvd/training_set/bfvd_data_3di.fasta
  test_3di_masked_fasta: /mnt/data1/bfvd/training_set/bfvd_data_3di_masked.fasta
  batch_size: 8
  mask_char: X
  output_dir: evaluation_results

✓ Output directory: evaluation_results


## 4. Load Test Data

Load the test sequences and ground truth 3Di labels.

In [58]:
# Load test amino acid sequences
print("Loading test data...")
test_aa_records = read_fasta(EVAL_CONFIG['test_aa_fasta'])
print(f"✓ Loaded {len(test_aa_records)} amino acid sequences")

# Load ground truth 3Di sequences (non-masked)
test_3di_records = read_fasta(EVAL_CONFIG['test_3di_fasta'])
print(f"✓ Loaded {len(test_3di_records)} 3Di sequences (ground truth)")

# Load masked 3Di sequences (if available)
if os.path.exists(EVAL_CONFIG['test_3di_masked_fasta']):
	test_3di_masked_records = read_fasta(EVAL_CONFIG['test_3di_masked_fasta'])
	print(f"✓ Loaded {len(test_3di_masked_records)} 3Di sequences (masked)")
else:
	test_3di_masked_records = None
	print("  Note: No masked 3Di file found (this is optional)")

# Verify alignment
assert len(test_aa_records) == len(test_3di_records), "Mismatch between AA and 3Di sequences"

# Create DataFrame for easier handling
test_data = pd.DataFrame({
	'header': [h for h, _ in test_aa_records],
	'aa_sequence': [s for _, s in test_aa_records],
	'true_3di': [s for _, s in test_3di_records],
})

if test_3di_masked_records:
	test_data['true_3di_masked'] = [s for _, s in test_3di_masked_records]

# Show sample
print("\nSample sequences:")
print(test_data.head())

# Compute statistics
test_data['seq_length'] = test_data['aa_sequence'].str.len()
print(f"\nSequence length statistics:")
print(test_data['seq_length'].describe())

Loading test data...
✓ Loaded 351242 amino acid sequences
✓ Loaded 351242 3Di sequences (ground truth)
✓ Loaded 351242 3Di sequences (masked)

Sample sequences:
       header                                        aa_sequence  \
0  A0A023GZ41  PSLQHPTFIAGKKCRAGYTYTSLDVRPTRTEKDKSFGQRLIIPVPV...   
1  A0A023H4N3  QSLGTPLSLSSSNGLGARFLYSFLKDFAAPRLLEEDLVFRMVFSIT...   
2  A0A023H4R8  KKVLTRVTEVAFRQFRPNSDAHSAIQSIATMLSSSTNHTIIGGVTL...   
3  A0A023H4Z6  EIIPDVSLTKPYEAVISGNDWMTLGRMIPTAPIPTIRDVFFSGLSR...   
4  A0A023H582  DEVSKINSMSILGPNQLKLCTQLVLSNGAAPVVLSLVSKEKKSILN...   

                                            true_3di  \
0  DPPVVVVCPPPPDADVQKDKDKDKDFFAKAFAQFKAKDFDDGDPPQ...   
1  DDPPPWDWQFDPPCVSLVVQLVVCCVVVPPPDPDPVNSNVVSVLVP...   
2  DDQLVPLLVVLLVPDDLPDDLVVSLVVSLVSLVVSQDQDDDPNHRD...   
3  DPDDDVPPCPPPDFPLWAQDPVVSDIDDGPDDNDDPVCVVVVCCPV...   
4  DVVVVVVVVLVPDPQPWDKDDKDWAFAPDAKDKDFPDDPVLLVVLV...   

                                     true_3di_masked  
0  XXXXXXVCPPPPDADVQKDKDKDKDFFAKAFAQFK

## 5. Find Available Checkpoints

In [43]:
# Find all checkpoint files
checkpoint_dir = Path(EVAL_CONFIG['checkpoint_dir'])
all_checkpoints = sorted(checkpoint_dir.glob('epoch_*.pt'))

print(f"Found {len(all_checkpoints)} checkpoints in {checkpoint_dir}")

# Filter by requested epochs if specified
if EVAL_CONFIG['epochs_to_eval'] is not None:
	checkpoints_to_eval = [
		cp for cp in all_checkpoints 
		if any(f'epoch_{epoch}.pt' in cp.name for epoch in EVAL_CONFIG['epochs_to_eval'])
	]
else:
	checkpoints_to_eval = all_checkpoints

print(f"\nCheckpoints to evaluate:")
for cp in checkpoints_to_eval:
	print(f"  - {cp.name}")

if not checkpoints_to_eval:
	print("\n⚠ No checkpoints found to evaluate!")
	print(f"   Please check the checkpoint directory: {checkpoint_dir}")

Found 3 checkpoints in checkpoints

Checkpoints to evaluate:
  - epoch_1.pt
  - epoch_2.pt
  - epoch_3.pt


## 6. Evaluation Metrics Functions

In [44]:
def calculate_accuracy(predictions: List[str], ground_truth: List[str], 
					  mask_char: str = 'X') -> Dict:
	"""
	Calculate various accuracy metrics.
	
	Args:
		predictions: List of predicted 3Di sequences
		ground_truth: List of true 3Di sequences
		mask_char: Character indicating masked positions (to ignore)
		
	Returns:
		Dictionary with accuracy metrics
	"""
	total_residues = 0
	correct_residues = 0
	total_sequences = len(predictions)
	perfect_sequences = 0
	
	per_seq_accuracies = []
	all_true_labels = []
	all_pred_labels = []
	
	for pred, true in zip(predictions, ground_truth):
		# Handle length mismatches
		min_len = min(len(pred), len(true))
		if len(pred) != len(true):
			print(f"Warning: Length mismatch - pred: {len(pred)}, true: {len(true)}")
		
		seq_correct = 0
		seq_total = 0
		
		for p, t in zip(pred[:min_len], true[:min_len]):
			# Skip masked positions
			if t == mask_char:
				continue
			
			seq_total += 1
			total_residues += 1
			
			if p == t:
				seq_correct += 1
				correct_residues += 1
			
			# Collect for confusion matrix
			all_true_labels.append(t)
			all_pred_labels.append(p)
		
		# Per-sequence accuracy
		if seq_total > 0:
			seq_acc = seq_correct / seq_total
			per_seq_accuracies.append(seq_acc)
			
			if seq_acc == 1.0:
				perfect_sequences += 1
	
	# Calculate overall metrics
	overall_accuracy = correct_residues / total_residues if total_residues > 0 else 0.0
	perfect_seq_percentage = perfect_sequences / total_sequences if total_sequences > 0 else 0.0
	mean_seq_accuracy = np.mean(per_seq_accuracies) if per_seq_accuracies else 0.0
	
	return {
		'overall_accuracy': overall_accuracy,
		'total_residues': total_residues,
		'correct_residues': correct_residues,
		'mean_sequence_accuracy': mean_seq_accuracy,
		'perfect_sequences': perfect_sequences,
		'perfect_sequence_percentage': perfect_seq_percentage,
		'per_sequence_accuracies': per_seq_accuracies,
		'true_labels': all_true_labels,
		'pred_labels': all_pred_labels,
	}

print("✓ Accuracy calculation function defined")

✓ Accuracy calculation function defined


## 7. Run Evaluation on All Checkpoints

This cell will evaluate all selected checkpoints and collect results.

In [53]:
from ESM3di_model import ESM3DiModel

In [54]:
# Store results for all checkpoints
all_results = []



print(f"Evaluating {len(checkpoints_to_eval)} checkpoints...\n")
print("=" * 80)

for checkpoint_path in checkpoints_to_eval:
	epoch_name = checkpoint_path.stem  # e.g., 'epoch_3'
	print(f"\n{'='*80}")
	print(f"EVALUATING: {epoch_name}")
	print(f"{'='*80}\n")
	
	# Load checkpoint
	model, tokenizer, label_vocab, mask_chars, config = load_checkpoint(
		str(checkpoint_path), device
	)
	
	# Run predictions
	print("Running predictions on test set...")
	aa_sequences = test_data['aa_sequence'].tolist()
	
	predictions = predict_3di_batch(
		model, tokenizer, label_vocab, aa_sequences,
		device=device, batch_size=EVAL_CONFIG['batch_size']
	)
	print(f"✓ Generated {len(predictions)} predictions\n")
	
	# Calculate accuracy metrics
	print("Calculating accuracy metrics...")
	metrics = calculate_accuracy(
		predictions, 
		test_data['true_3di'].tolist(),
		mask_char=EVAL_CONFIG['mask_char']
	)
	
	# Print summary
	print(f"\n{'='*80}")
	print(f"RESULTS: {epoch_name}")
	print(f"{'='*80}")
	print(f"Overall Accuracy:           {metrics['overall_accuracy']*100:.2f}%")
	print(f"Mean Sequence Accuracy:     {metrics['mean_sequence_accuracy']*100:.2f}%")
	print(f"Perfect Sequences:          {metrics['perfect_sequences']}/{len(test_data)} "
		  f"({metrics['perfect_sequence_percentage']*100:.2f}%)")
	print(f"Total Residues Evaluated:   {metrics['total_residues']:,}")
	print(f"Correct Predictions:        {metrics['correct_residues']:,}")
	print(f"{'='*80}\n")
	
	# Store results
	result = {
		'checkpoint': epoch_name,
		'checkpoint_path': str(checkpoint_path),
		'predictions': predictions,
		'metrics': metrics,
		'label_vocab': label_vocab,
	}
	all_results.append(result)
	
	# Clean up
	del model
	torch.cuda.empty_cache()

print(f"\n{'='*80}")
print("EVALUATION COMPLETE")
print(f"{'='*80}")

Evaluating 3 checkpoints...


EVALUATING: epoch_1

Loading checkpoint: checkpoints/epoch_1.pt
  Checkpoint: epoch 0, loss: 0.0000
  Label vocab size: 20
  Mask characters: X
  Base model: facebook/esm2_t12_35M_UR50D
loading fine tuned  facebook/esm2_t12_35M_UR50D


/tmp/ipykernel_10626/3462448501.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location=device)
/home/dmoi/miniforge3/envs/pyg/l


Loading base model: facebook/esm2_t12_35M_UR50D


Some weights of EsmForTokenClassification were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Base model loaded

Auto-discovering LoRA target modules...
Discovered target modules: ['query', 'key', 'value', 'dense']
✓ LoRA setup complete



RuntimeError: Error(s) in loading state_dict for PeftModelForTokenClassification:
	Missing key(s) in state_dict: "base_model.model.esm.embeddings.position_embeddings.weight". 

## 8. Compare Results Across Checkpoints

In [ ]:
# Create comparison DataFrame
comparison_data = []

for result in all_results:
	metrics = result['metrics']
	comparison_data.append({
		'Checkpoint': result['checkpoint'],
		'Overall Accuracy (%)': metrics['overall_accuracy'] * 100,
		'Mean Sequence Accuracy (%)': metrics['mean_sequence_accuracy'] * 100,
		'Perfect Sequences (%)': metrics['perfect_sequence_percentage'] * 100,
		'Total Residues': metrics['total_residues'],
		'Correct Residues': metrics['correct_residues'],
	})

comparison_df = pd.DataFrame(comparison_data)

print("\nPerformance Comparison:")
print(comparison_df.to_string(index=False))

# Save to CSV
output_csv = os.path.join(EVAL_CONFIG['output_dir'], 'checkpoint_comparison.csv')
comparison_df.to_csv(output_csv, index=False)
print(f"\n✓ Saved comparison to {output_csv}")

## 9. Visualize Performance Across Checkpoints

In [ ]:
# Plot accuracy metrics across checkpoints
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall Accuracy
axes[0].plot(comparison_df.index, comparison_df['Overall Accuracy (%)'], 
			 'o-', linewidth=2, markersize=10, color='steelblue')
axes[0].set_xlabel('Checkpoint')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Overall Per-Residue Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xticks(comparison_df.index)
axes[0].set_xticklabels(comparison_df['Checkpoint'], rotation=45)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 100])

# Mean Sequence Accuracy
axes[1].plot(comparison_df.index, comparison_df['Mean Sequence Accuracy (%)'], 
			 'o-', linewidth=2, markersize=10, color='forestgreen')
axes[1].set_xlabel('Checkpoint')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Mean Sequence Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xticks(comparison_df.index)
axes[1].set_xticklabels(comparison_df['Checkpoint'], rotation=45)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 100])

# Perfect Sequences
axes[2].plot(comparison_df.index, comparison_df['Perfect Sequences (%)'], 
			 'o-', linewidth=2, markersize=10, color='crimson')
axes[2].set_xlabel('Checkpoint')
axes[2].set_ylabel('Perfect Sequences (%)')
axes[2].set_title('Perfect Sequence Predictions', fontsize=14, fontweight='bold')
axes[2].set_xticks(comparison_df.index)
axes[2].set_xticklabels(comparison_df['Checkpoint'], rotation=45)
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([0, 100])

plt.tight_layout()
plot_path = os.path.join(EVAL_CONFIG['output_dir'], 'checkpoint_comparison.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✓ Saved plot to {plot_path}")
plt.show()

## 10. Per-Sequence Accuracy Distribution

Visualize the distribution of per-sequence accuracies for each checkpoint.

In [ ]:
# Plot per-sequence accuracy distributions
fig, axes = plt.subplots(1, len(all_results), figsize=(6*len(all_results), 5))

if len(all_results) == 1:
	axes = [axes]

for idx, result in enumerate(all_results):
	per_seq_acc = np.array(result['metrics']['per_sequence_accuracies']) * 100
	
	axes[idx].hist(per_seq_acc, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
	axes[idx].axvline(per_seq_acc.mean(), color='red', linestyle='--', 
					 linewidth=2, label=f'Mean: {per_seq_acc.mean():.1f}%')
	axes[idx].set_xlabel('Accuracy (%)', fontsize=12)
	axes[idx].set_ylabel('Number of Sequences', fontsize=12)
	axes[idx].set_title(f"{result['checkpoint']}\nPer-Sequence Accuracy Distribution", 
					   fontsize=13, fontweight='bold')
	axes[idx].legend(fontsize=11)
	axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(EVAL_CONFIG['output_dir'], 'sequence_accuracy_distributions.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✓ Saved plot to {plot_path}")
plt.show()

## 11. Confusion Matrix (Best Checkpoint)

Generate a confusion matrix for the best performing checkpoint.

In [ ]:
# Find best checkpoint (by overall accuracy)
best_idx = np.argmax([r['metrics']['overall_accuracy'] for r in all_results])
best_result = all_results[best_idx]

print(f"Best checkpoint: {best_result['checkpoint']}")
print(f"Overall accuracy: {best_result['metrics']['overall_accuracy']*100:.2f}%\n")

# Get labels
true_labels = best_result['metrics']['true_labels']
pred_labels = best_result['metrics']['pred_labels']
label_vocab = best_result['label_vocab']

# Compute confusion matrix
cm = confusion_matrix(true_labels, pred_labels, labels=label_vocab)

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
			xticklabels=label_vocab, yticklabels=label_vocab,
			cbar_kws={'label': 'Count'})
plt.xlabel('Predicted 3Di', fontsize=13, fontweight='bold')
plt.ylabel('True 3Di', fontsize=13, fontweight='bold')
plt.title(f'Confusion Matrix - {best_result["checkpoint"]}\n'
		  f'Overall Accuracy: {best_result["metrics"]["overall_accuracy"]*100:.2f}%',
		  fontsize=14, fontweight='bold')
plt.tight_layout()

plot_path = os.path.join(EVAL_CONFIG['output_dir'], f'confusion_matrix_{best_result["checkpoint"]}.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✓ Saved confusion matrix to {plot_path}")
plt.show()

## 12. Per-Class Performance (Best Checkpoint)

In [ ]:
# Generate classification report
print(f"Classification Report - {best_result['checkpoint']}")
print("=" * 80)

report = classification_report(
	true_labels, pred_labels, 
	labels=label_vocab,
	target_names=label_vocab,
	zero_division=0
)
print(report)

# Save report
report_path = os.path.join(EVAL_CONFIG['output_dir'], 
						  f'classification_report_{best_result["checkpoint"]}.txt')
with open(report_path, 'w') as f:
	f.write(f"Classification Report - {best_result['checkpoint']}\n")
	f.write("=" * 80 + "\n")
	f.write(report)

print(f"\n✓ Saved classification report to {report_path}")

## 13. Visualize Example Predictions

Show detailed predictions for a few example sequences.

In [ ]:
# Select checkpoint to visualize (use best checkpoint)
selected_result = best_result
predictions = selected_result['predictions']

# Number of examples to show
num_examples = min(5, len(test_data))

print(f"Example Predictions - {selected_result['checkpoint']}")
print("=" * 100)

for i in range(num_examples):
	header = test_data.iloc[i]['header']
	aa_seq = test_data.iloc[i]['aa_sequence']
	true_3di = test_data.iloc[i]['true_3di']
	pred_3di = predictions[i]
	
	# Calculate accuracy for this sequence
	matches = sum(1 for p, t in zip(pred_3di, true_3di) 
				 if t != EVAL_CONFIG['mask_char'] and p == t)
	valid_pos = sum(1 for t in true_3di if t != EVAL_CONFIG['mask_char'])
	accuracy = (matches / valid_pos * 100) if valid_pos > 0 else 0.0
	
	print(f"\nExample {i+1}: {header}")
	print(f"Length: {len(aa_seq)} residues | Accuracy: {accuracy:.2f}%")
	print("-" * 100)
	
	# Show first 80 characters
	show_len = min(80, len(aa_seq))
	print(f"AA:        {aa_seq[:show_len]}")
	print(f"True 3Di:  {true_3di[:show_len]}")
	print(f"Predicted: {pred_3di[:show_len]}")
	
	# Show match/mismatch indicators
	indicators = ''.join(['✓' if p == t and t != EVAL_CONFIG['mask_char'] 
						 else '✗' if p != t and t != EVAL_CONFIG['mask_char']
						 else '-'
						 for p, t in zip(pred_3di[:show_len], true_3di[:show_len])])
	print(f"Match:     {indicators}")
	
	if len(aa_seq) > show_len:
		print(f"... (showing first {show_len} of {len(aa_seq)} residues)")

print("\n" + "=" * 100)

## 14. Test on Custom Sequences

You can test the model on your own amino acid sequences here.

In [ ]:
# Example: Test on custom sequences
custom_sequences = [
	("Custom_seq_1", "MKTAYIAKQRQISFVKSHFSRQLEASDKLLDDQYDFTVL"),
	("Custom_seq_2", "MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCL"),
	# Add your own sequences here
]

if custom_sequences:
	print("Testing on custom sequences...\n")
	
	# Use best checkpoint
	model, tokenizer, label_vocab, mask_chars, config = load_checkpoint(
		best_result['checkpoint_path'], device
	)
	
	# Run predictions
	custom_aa_seqs = [seq for _, seq in custom_sequences]
	custom_predictions = predict_3di_batch(
		model, tokenizer, label_vocab, custom_aa_seqs,
		device=device, batch_size=EVAL_CONFIG['batch_size']
	)
	
	# Display results
	print("Custom Sequence Predictions")
	print("=" * 80)
	
	for (header, aa_seq), pred_3di in zip(custom_sequences, custom_predictions):
		print(f"\n{header}")
		print(f"AA sequence: {aa_seq}")
		print(f"Predicted 3Di: {pred_3di}")
		print(f"Length: {len(aa_seq)} residues")
	
	print("\n" + "=" * 80)
	
	# Clean up
	del model
	torch.cuda.empty_cache()
else:
	print("No custom sequences defined. Add sequences to the 'custom_sequences' list above.")

## 15. Export Detailed Results

Export all predictions and metrics to files for further analysis.

In [ ]:
# Export detailed predictions for best checkpoint
predictions_df = test_data.copy()
predictions_df['predicted_3di'] = best_result['predictions']

# Calculate per-sequence accuracy
def calc_seq_accuracy(row):
	pred = row['predicted_3di']
	true = row['true_3di']
	matches = sum(1 for p, t in zip(pred, true) 
				 if t != EVAL_CONFIG['mask_char'] and p == t)
	valid = sum(1 for t in true if t != EVAL_CONFIG['mask_char'])
	return (matches / valid * 100) if valid > 0 else 0.0

predictions_df['accuracy_%'] = predictions_df.apply(calc_seq_accuracy, axis=1)

# Save to CSV
predictions_csv = os.path.join(EVAL_CONFIG['output_dir'], 
							  f'predictions_{best_result["checkpoint"]}.csv')
predictions_df.to_csv(predictions_csv, index=False)
print(f"✓ Saved detailed predictions to {predictions_csv}")

# Save summary statistics
summary = {
	'checkpoint': best_result['checkpoint'],
	'checkpoint_path': best_result['checkpoint_path'],
	'test_sequences': len(test_data),
	'overall_accuracy': best_result['metrics']['overall_accuracy'],
	'mean_sequence_accuracy': best_result['metrics']['mean_sequence_accuracy'],
	'perfect_sequences': best_result['metrics']['perfect_sequences'],
	'perfect_sequence_percentage': best_result['metrics']['perfect_sequence_percentage'],
	'total_residues': best_result['metrics']['total_residues'],
	'correct_residues': best_result['metrics']['correct_residues'],
	'label_vocabulary': best_result['label_vocab'],
}

summary_json = os.path.join(EVAL_CONFIG['output_dir'], 'evaluation_summary.json')
with open(summary_json, 'w') as f:
	json.dump(summary, f, indent=2)
print(f"✓ Saved summary to {summary_json}")

print(f"\n{'='*80}")
print("All evaluation results exported successfully!")
print(f"{'='*80}")
print(f"\nResults directory: {EVAL_CONFIG['output_dir']}")
print("\nGenerated files:")
for file in sorted(Path(EVAL_CONFIG['output_dir']).glob('*')):
	print(f"  - {file.name}")

## 16. Summary and Recommendations

Based on the evaluation results, here are some insights and next steps.

In [ ]:
print("="*80)
print("EVALUATION SUMMARY")
print("="*80)

best_checkpoint = best_result['checkpoint']
best_accuracy = best_result['metrics']['overall_accuracy'] * 100

print(f"\nBest Checkpoint: {best_checkpoint}")
print(f"Best Overall Accuracy: {best_accuracy:.2f}%")
print(f"\nAll evaluated checkpoints:")
for result in all_results:
	acc = result['metrics']['overall_accuracy'] * 100
	status = "⭐ BEST" if result == best_result else ""
	print(f"  {result['checkpoint']}: {acc:.2f}% {status}")

print(f"\n{'='*80}")
print("RECOMMENDATIONS")
print("="*80)

if best_accuracy < 70:
	print("\n📊 Model Performance: NEEDS IMPROVEMENT")
	print("\nSuggestions:")
	print("  1. Train for more epochs")
	print("  2. Increase training data size")
	print("  3. Try a larger ESM model (e.g., esm2_t33_650M_UR50D)")
	print("  4. Adjust LoRA hyperparameters (increase rank)")
	print("  5. Reduce learning rate for more stable training")
elif best_accuracy < 85:
	print("\n📊 Model Performance: MODERATE")
	print("\nSuggestions:")
	print("  1. Continue training for more epochs")
	print("  2. Add more diverse training sequences")
	print("  3. Consider fine-tuning learning rate")
	print("  4. Evaluate on a separate validation set")
else:
	print("\n📊 Model Performance: EXCELLENT")
	print("\nNext Steps:")
	print("  1. Test on external/independent datasets")
	print("  2. Deploy model for production use")
	print("  3. Create FoldSeek database with trained model")
	print("  4. Consider publishing or sharing results")

print(f"\n{'='*80}")
print("\n✓ Evaluation complete! Check the output directory for all results and plots.")
print(f"  Output directory: {EVAL_CONFIG['output_dir']}")